# Quantitative Evaluation: Fine-Tuned Qwen2.5-VL-7B

**Goal**: Evaluate the QLoRA fine-tuned model on the held-out test set (14 samples) using:

| Metric | What It Measures |
|--------|------------------|
| **JSON Schema Compliance** | % of outputs that parse as valid JSON matching the target schema |
| **Scene Type Accuracy** | Exact-match classification accuracy |
| **ROUGE-L** | N-gram overlap between generated and reference captions |
| **BERTScore** | Semantic similarity between generated and reference captions |
| **Object Mention F1** | Precision/recall on predicted object types vs. ground truth |

**Setup**: Load the fine-tuned LoRA adapter from Drive, run inference on the test split, compute all metrics.

In [ ]:
%%capture
!pip install "transformers>=4.49" peft bitsandbytes qwen-vl-utils accelerate \
    rouge-score bert-score scikit-learn matplotlib

In [ ]:
import json, textwrap, time
from pathlib import Path
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from google.colab import drive

# ── Config ──────────────────────────────────────────────────────
QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
DRIVE_PROJECT = Path("/content/drive/MyDrive/vlm-scene-analyser")
ADAPTER_PATH = DRIVE_PROJECT / "models" / "qwen_lora_v1"
TEST_SPLIT_PATH = ADAPTER_PATH / "test_split.jsonl"
IMAGE_DIR = Path("/content/images/nadir")

SYSTEM_PROMPT = (
    "You are an aerial scene analyst specialising in Singapore urban landscapes. "
    "Given a nadir (top-down) aerial image, produce a JSON object with: "
    "caption (2-4 sentences using Singapore-specific vocabulary like HDB block, "
    "hawker centre, void deck, covered walkway, MRT), scene_type, objects with counts, "
    "infrastructure, terrain, and notable features."
)
USER_PROMPT = "Analyse this nadir aerial image of Singapore."

# Valid scene types and object types for evaluation
VALID_SCENE_TYPES = {
    "residential_hdb", "residential_private", "commercial_cbd", "industrial",
    "port_terminal", "airport", "park_green", "waterway", "construction",
    "military", "mixed_use", "transport_hub",
}

VALID_OBJECT_TYPES = {
    "hdb_block", "condo", "landed_house", "hawker_centre", "mrt_station",
    "bus_interchange", "shopping_mall", "warehouse", "container_crane",
    "cargo_ship", "aircraft", "vehicle", "construction_crane", "solar_panel",
    "sports_facility", "place_of_worship", "school",
}

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Load fine-tuned model ────────────────────────────────────
import shutil
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

drive.mount("/content/drive")

# Copy images locally for faster I/O
if not IMAGE_DIR.exists():
    IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    for f in (DRIVE_PROJECT / "images" / "nadir").glob("*.jpg"):
        shutil.copy2(f, IMAGE_DIR / f.name)

# Load base model (4-bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading {QWEN_MODEL_ID} (4-bit NF4)...")
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(
    QWEN_MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28,
)

# Load LoRA adapter
print(f"Loading adapter from {ADAPTER_PATH}...")
model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))
model.eval()
print("Model loaded.")

# Load test split
test_ann = []
with open(TEST_SPLIT_PATH) as f:
    for line in f:
        ann = json.loads(line)
        ann["image_path"] = str(IMAGE_DIR / ann["image_file"])
        test_ann.append(ann)

print(f"Test set: {len(test_ann)} samples")
print(f"Scene types: {Counter(a['scene_type'] for a in test_ann)}")

In [ ]:
# ── Run inference on test set ────────────────────────────────
from qwen_vl_utils import process_vision_info

predictions = []

for i, ann in enumerate(test_ann):
    image = Image.open(ann["image_path"]).convert("RGB")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": USER_PROMPT},
        ]},
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    img_inputs, _ = process_vision_info(messages)
    inputs = processor(
        text=[text], images=img_inputs, padding=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    gen_trimmed = gen[:, inputs["input_ids"].shape[1]:]
    pred_text = processor.batch_decode(gen_trimmed, skip_special_tokens=True)[0]

    predictions.append({
        "image_file": ann["image_file"],
        "scene_type_gt": ann["scene_type"],
        "caption_gt": ann["caption"],
        "analysis_gt": ann["analysis"],
        "raw_prediction": pred_text,
    })

    print(f"[{i+1}/{len(test_ann)}] {ann['image_file'][:40]}... done")

print(f"\nGenerated {len(predictions)} predictions.")

In [ ]:
# ── Metric 1: JSON Schema Compliance ─────────────────────────

def validate_schema(raw_text):
    """Check if raw prediction is valid JSON with required schema fields."""
    try:
        obj = json.loads(raw_text)
    except json.JSONDecodeError:
        return None, "invalid_json"

    errors = []
    if "caption" not in obj:
        errors.append("missing_caption")
    if "scene_type" not in obj:
        errors.append("missing_scene_type")

    analysis = obj.get("analysis", {})
    if not isinstance(analysis, dict):
        errors.append("analysis_not_dict")
    else:
        for field in ["objects", "infrastructure", "terrain", "notable"]:
            if field not in analysis:
                errors.append(f"missing_analysis.{field}")

    if errors:
        return obj, ", ".join(errors)
    return obj, None


parsed_preds = []
schema_results = []

for pred in predictions:
    obj, error = validate_schema(pred["raw_prediction"])
    parsed_preds.append(obj)
    schema_results.append({
        "image": pred["image_file"],
        "valid": error is None,
        "error": error,
    })

n_valid = sum(1 for r in schema_results if r["valid"])
compliance_rate = n_valid / len(schema_results) * 100

print(f"Schema Compliance: {n_valid}/{len(schema_results)} ({compliance_rate:.1f}%)")
for r in schema_results:
    status = "OK" if r["valid"] else r["error"]
    print(f"  {r['image'][:45]:45s} {status}")

In [ ]:
# ── Metric 2: Scene Type Accuracy ────────────────────────────

correct = 0
total = 0
confusion = defaultdict(Counter)

for pred, parsed in zip(predictions, parsed_preds):
    gt = pred["scene_type_gt"]
    if parsed is not None:
        predicted = parsed.get("scene_type", "<missing>")
    else:
        predicted = "<parse_fail>"

    confusion[gt][predicted] += 1
    if predicted == gt:
        correct += 1
    total += 1

accuracy = correct / total * 100
print(f"Scene Type Accuracy: {correct}/{total} ({accuracy:.1f}%)\n")

print(f"{'Ground Truth':25s} {'Predicted':25s} Count")
print("-" * 60)
for gt in sorted(confusion):
    for predicted, count in confusion[gt].most_common():
        marker = "  ✓" if gt == predicted else "  ✗"
        print(f"{gt:25s} {predicted:25s} {count}{marker}")

In [ ]:
# ── Metric 3: ROUGE-L on captions ────────────────────────────
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

rouge_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

for pred, parsed in zip(predictions, parsed_preds):
    gt_caption = pred["caption_gt"]
    if parsed is not None and "caption" in parsed:
        pred_caption = parsed["caption"]
    else:
        pred_caption = ""  # No valid caption generated

    scores = scorer.score(gt_caption, pred_caption)
    for key in rouge_scores:
        rouge_scores[key].append(scores[key].fmeasure)

print("ROUGE Scores (F1):")
for key in ["rouge1", "rouge2", "rougeL"]:
    vals = rouge_scores[key]
    print(f"  {key:8s}  mean={np.mean(vals):.4f}  std={np.std(vals):.4f}  "
          f"min={np.min(vals):.4f}  max={np.max(vals):.4f}")

# Per-sample breakdown
print(f"\n{'Image':45s} ROUGE-L")
print("-" * 55)
for pred, rl in zip(predictions, rouge_scores["rougeL"]):
    print(f"{pred['image_file'][:45]:45s} {rl:.4f}")

In [ ]:
# ── Metric 4: BERTScore on captions ──────────────────────────
from bert_score import score as bert_score_fn

gt_captions = [p["caption_gt"] for p in predictions]
pred_captions = [
    parsed["caption"] if parsed is not None and "caption" in parsed else ""
    for parsed in parsed_preds
]

P, R, F1 = bert_score_fn(
    pred_captions, gt_captions, lang="en", verbose=True, device=model.device
)

print(f"\nBERTScore (F1):")
print(f"  mean={F1.mean():.4f}  std={F1.std():.4f}  "
      f"min={F1.min():.4f}  max={F1.max():.4f}")

print(f"\n{'Image':45s} P       R       F1")
print("-" * 75)
for pred, p, r, f in zip(predictions, P, R, F1):
    print(f"{pred['image_file'][:45]:45s} {p:.4f}  {r:.4f}  {f:.4f}")

In [ ]:
# ── Metric 5: Object Mention F1 ──────────────────────────────
# Custom metric: precision/recall on predicted object types vs. ground truth.
# Evaluates whether the model mentions the right categories of objects,
# independent of exact counts.


def extract_object_types(analysis):
    """Extract set of object type labels from an analysis dict."""
    if not isinstance(analysis, dict):
        return set()
    objects = analysis.get("objects", [])
    if not isinstance(objects, list):
        return set()
    return {obj["type"] for obj in objects if isinstance(obj, dict) and "type" in obj}


per_sample_f1 = []
per_type_tp = Counter()
per_type_fp = Counter()
per_type_fn = Counter()

for pred, parsed in zip(predictions, parsed_preds):
    gt_types = extract_object_types(pred["analysis_gt"])

    if parsed is not None:
        pred_types = extract_object_types(parsed.get("analysis", {}))
    else:
        pred_types = set()

    tp = gt_types & pred_types
    fp = pred_types - gt_types
    fn = gt_types - pred_types

    precision = len(tp) / len(pred_types) if pred_types else 0.0
    recall = len(tp) / len(gt_types) if gt_types else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    per_sample_f1.append(f1)

    for t in tp:
        per_type_tp[t] += 1
    for t in fp:
        per_type_fp[t] += 1
    for t in fn:
        per_type_fn[t] += 1

# Aggregate metrics
mean_f1 = np.mean(per_sample_f1)
print(f"Object Mention F1 (macro): {mean_f1:.4f}\n")

# Per-type breakdown
all_types = sorted(set(per_type_tp) | set(per_type_fp) | set(per_type_fn))
print(f"{'Object Type':25s} {'TP':>4s} {'FP':>4s} {'FN':>4s} {'Prec':>6s} {'Rec':>6s} {'F1':>6s}")
print("-" * 75)
for t in all_types:
    tp = per_type_tp[t]
    fp = per_type_fp[t]
    fn = per_type_fn[t]
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0
    print(f"{t:25s} {tp:4d} {fp:4d} {fn:4d} {p:6.2f} {r:6.2f} {f:6.2f}")

# Per-sample breakdown
print(f"\n{'Image':45s} F1")
print("-" * 55)
for pred, f1 in zip(predictions, per_sample_f1):
    print(f"{pred['image_file'][:45]:45s} {f1:.4f}")

In [ ]:
# ── Summary ──────────────────────────────────────────────────

print("=" * 60)
print("  EVALUATION SUMMARY — Fine-Tuned Qwen2.5-VL-7B")
print("=" * 60)
print(f"  Test set size:          {len(predictions)}")
print(f"  Schema Compliance:      {compliance_rate:.1f}%")
print(f"  Scene Type Accuracy:    {accuracy:.1f}%")
print(f"  ROUGE-1 F1:             {np.mean(rouge_scores['rouge1']):.4f}")
print(f"  ROUGE-2 F1:             {np.mean(rouge_scores['rouge2']):.4f}")
print(f"  ROUGE-L F1:             {np.mean(rouge_scores['rougeL']):.4f}")
print(f"  BERTScore F1:           {F1.mean():.4f}")
print(f"  Object Mention F1:      {mean_f1:.4f}")
print("=" * 60)

# Export results
results = {
    "n_test": len(predictions),
    "schema_compliance_pct": round(compliance_rate, 1),
    "scene_type_accuracy_pct": round(accuracy, 1),
    "rouge1_f1": round(float(np.mean(rouge_scores["rouge1"])), 4),
    "rouge2_f1": round(float(np.mean(rouge_scores["rouge2"])), 4),
    "rougeL_f1": round(float(np.mean(rouge_scores["rougeL"])), 4),
    "bertscore_f1": round(float(F1.mean()), 4),
    "object_mention_f1": round(float(mean_f1), 4),
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nResults saved to evaluation_results.json")